# [Kaggle] Pipeline: ATE + ASC & Full Evaluation
Can chay NB01 va NB02 truoc de co models.

**Evaluation**:
- Span-Level Exact Match F1 (nghiem ngat)
- Sentence-Level Multi-Label F1 (giong `phobert-crf-absa.ipynb`)


## 0. Setup


In [ ]:
import subprocess, sys, os

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'underthesea', 'pytorch-crf', 'gensim'])

DATA_DATASET = "UIT-ViSD4SA"
SRC_DATASET  = "absa-src"

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/datasets/danghoang1302/uit-visd4sa'
    SAVE_DIR = '/kaggle/working/results'
    SRC_INPUT = '/kaggle/input/datasets/danghoang1302/absa-src'
    os.system(f'cp -r {SRC_INPUT}/src /kaggle/working/src')
    sys.path.insert(0, '/kaggle/working')
    print(f"KAGGLE | Data: {DATA_DIR}")
else:
    sys.path.insert(0, os.path.abspath(".."))
    DATA_DIR = os.path.join("..", "..", "data")
    SAVE_DIR = os.path.join("..", "..", "results")
    print(f"LOCAL mode")

for fn in ['train.jsonl', 'dev.jsonl', 'test.jsonl']:
    assert os.path.exists(os.path.join(DATA_DIR, fn)), f"MISSING: {fn}"
print("All data files OK!")


## 1. Imports & Load Models


In [ ]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import (load_raw_data, segment_items, build_vocab,
                                  train_w2v_embeddings, tokenize_baseline)
from src.ate.ate_dataset import ATEDataset, BIO_TAGS, TAG2ID, NUM_TAGS, ASPECTS
from src.ate.ate_model import build_ate_model
from src.asc.asc_model import build_asc_model
from src.utils.metrics import (bio_tags_to_spans, evaluate_spans_f1,
                               char_spans_to_word_spans,
                               bio_to_sentence_labels, evaluate_multilabel)
from src.utils.visualization import plot_error_analysis

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print("Building vocab & embeddings from scratch...")
train_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "train.jsonl")))
dev_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "dev.jsonl")))
test_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "test.jsonl")))

all_texts = [item["text"] for item in train_items + dev_items + test_items]
word2idx = build_vocab(all_texts, min_freq=2, special_tokens=["[ASP]"])
VOCAB_SIZE = len(word2idx)
EMB_DIM = 150
emb_matrix = train_w2v_embeddings(all_texts, word2idx, emb_dim=EMB_DIM)
print(f"Vocab: {VOCAB_SIZE} | Emb: {EMB_DIM}d")

MODELS_DIR = "/kaggle/input/datasets/danghoang1302/models"
best_ate = "BiLSTM-CRF"
best_asc = "BiGRU"

try:
    print(f"Loading ATE: {best_ate} | ASC: {best_asc} tu {MODELS_DIR}")
    ate_type = best_ate.replace("-CRF", "")
    ate_model = build_ate_model(model_type=ate_type, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM,
        hidden_dim=256, num_tags=NUM_TAGS, n_layers=2, dropout=0.3).to(device)
    ate_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, f"best_ate_{best_ate}.pt"), map_location=device))
    ate_model.eval()

    asc_model = build_asc_model(model_type=best_asc, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM,
        hidden_dim=256, num_classes=3, n_layers=2, dropout=0.3).to(device)
    asc_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, f"best_asc_{best_asc}.pt"), map_location=device))
    asc_model.eval()
    print("Models loaded successfully!")
except Exception as e:
    print(f"Loi load model: {e}")


## 2. Pipeline Inference & Evaluation


In [ ]:
MAX_LEN = 128
SENTIMENT_NAMES = {0: "POSITIVE", 1: "NEGATIVE", 2: "NEUTRAL"}
LABEL_NAMES = [f"{a}#{s}" for a in ASPECTS for s in ["POSITIVE","NEUTRAL","NEGATIVE"]]

def pipeline_predict(text, ate_model, asc_model, word2idx, device):
    """Pipeline: ATE -> ASC. Returns list of (combined_label, start_word, end_word)."""
    words = text.split()[:MAX_LEN]
    seq, length = tokenize_baseline(text, word2idx, MAX_LEN)
    seq_t = torch.tensor([seq], dtype=torch.long).to(device)
    mask_t = torch.zeros(1, MAX_LEN, dtype=torch.bool); mask_t[0, :length] = True
    mask_t = mask_t.to(device); lens_t = torch.tensor([length])

    with torch.no_grad():
        pred_tags = ate_model(seq_t, mask=mask_t, lens=lens_t)[0]
    spans = bio_tags_to_spans(pred_tags, BIO_TAGS, length)

    results = []
    for aspect_label, start_idx, end_idx in spans:
        aspect_text = " ".join(words[start_idx:end_idx])
        marked = " ".join(words[:start_idx]) + " [ASP] " + aspect_text + " [ASP] " + " ".join(words[end_idx:])
        asc_seq, _ = tokenize_baseline(marked.strip(), word2idx, MAX_LEN)
        asc_t = torch.tensor([asc_seq], dtype=torch.long).to(device)
        with torch.no_grad():
            sentiment_id = asc_model(asc_t).argmax(dim=1).item()
        combined = f"{aspect_label}#{SENTIMENT_NAMES[sentiment_id]}"
        results.append((combined, start_idx, end_idx))
    return results

# === Full test set ===
print("Running pipeline on test set...")
all_true, all_pred = [], []
for item in test_items:
    true_spans = char_spans_to_word_spans(item["text"], item.get("labels", []), MAX_LEN)
    pred_spans = pipeline_predict(item["text"], ate_model, asc_model, word2idx, device)
    all_true.append(true_spans)
    all_pred.append(pred_spans)

# METRIC 1: Span-Level
span_res = evaluate_spans_f1(all_pred, all_true)
print(f"\n{'='*60}")
print(f"  METRIC 1: Span-Level Exact Match")
print(f"{'='*60}")
print(f"  P={span_res['precision']:.4f}  R={span_res['recall']:.4f}  F1={span_res['f1']:.4f}")
print(f"  (TP={span_res['tp']}, FP={span_res['fp']}, FN={span_res['fn']})")

# METRIC 2: Sentence-Level Multi-Label
label2id = {ln: i for i, ln in enumerate(LABEL_NAMES)}
num_labels = len(LABEL_NAMES)
true_sent, pred_sent = [], []
for ts, ps in zip(all_true, all_pred):
    tv = [0]*num_labels
    for l,_,_ in ts:
        if l in label2id: tv[label2id[l]] = 1
    pv = [0]*num_labels
    for l,_,_ in ps:
        if l in label2id: pv[label2id[l]] = 1
    true_sent.append(tv); pred_sent.append(pv)
mt = evaluate_multilabel(true_sent, pred_sent, LABEL_NAMES)

print(f"\n{'='*60}")
print(f"  METRIC 2: Sentence-Level Multi-Label")
print(f"{'='*60}")
for avg in ['micro','macro','weighted']:
    m = mt[avg]
    print(f"  {avg:<12} P={m['precision']:.4f}  R={m['recall']:.4f}  F1={m['f1']:.4f}")
print(f"\n  {'Label':<25} {'P':>7} {'R':>7} {'F1':>7} {'Sup':>6}")
print(f"  {'-'*55}")
for ln in LABEL_NAMES:
    m = mt[ln]
    print(f"  {ln:<25} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f1']:>7.4f} {m['support']:>6d}")


## 3. Demo: Test Predictions
Xem dau ra cua Pipeline tren 10 cau test.


In [ ]:
import random
random.seed(42)
sample_indices = random.sample(range(len(test_items)), min(10, len(test_items)))

print(f"{'='*70}")
print(f"  PIPELINE DEMO: 10 cau test ngau nhien")
print(f"{'='*70}")

for idx in sample_indices:
    item = test_items[idx]
    text = item["text"]
    words = text.split()[:MAX_LEN]

    true_spans = all_true[idx]
    pred_spans = all_pred[idx]
    true_set = set(true_spans)
    pred_set = set(pred_spans)

    tp = true_set & pred_set
    fp = pred_set - true_set
    fn = true_set - pred_set

    print(f"\n{'─'*70}")
    print(f"  [{idx}] {text[:100]}{'...' if len(text)>100 else ''}")
    print(f"  TRUE ({len(true_spans)}):")
    for label, s, e in true_spans:
        aspect_text = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = '✅' if (label, s, e) in pred_set else '❌ MISSED'
        print(f"    {label:<30} [{s}:{e}] \"{aspect_text}\"  {match}")

    print(f"  PRED ({len(pred_spans)}):")
    for label, s, e in pred_spans:
        aspect_text = ' '.join(words[s:e]) if e <= len(words) else '?'
        match = '✅' if (label, s, e) in true_set else '❌ WRONG'
        print(f"    {label:<30} [{s}:{e}] \"{aspect_text}\"  {match}")

    if not pred_spans:
        print(f"    (khong co prediction)")

print(f"\n{'='*70}")
print(f"  Tong: TP={len(tp)} correct, FP={len(fp)} wrong, FN={len(fn)} missed")


## 4. Error Analysis


In [ ]:
errors = []
for i, (ts, ps) in enumerate(zip(all_true, all_pred)):
    true_set = set(ts); pred_set = set(ps)
    for label, s, e in true_set - pred_set:
        errors.append({"Text": test_items[i]["text"][:100],
            "Error_Type": f"MISSED: {label} [{s}:{e}]", "Direction": "FN"})
    for label, s, e in pred_set - true_set:
        errors.append({"Text": test_items[i]["text"][:100],
            "Error_Type": f"WRONG: {label} [{s}:{e}]", "Direction": "FP"})

error_df = pd.DataFrame(errors)
print(f"Errors: {len(error_df)} (FN: {(error_df['Direction']=='FN').sum()}, FP: {(error_df['Direction']=='FP').sum()})")
display(error_df.head(15))
plot_error_analysis(error_df, top_n=15)

PIPE_DIR = os.path.join(SAVE_DIR, "pipeline"); os.makedirs(PIPE_DIR, exist_ok=True)
error_df.to_csv(os.path.join(PIPE_DIR, "error_analysis.csv"), index=False)
print(f"\nPipeline Span-F1={span_res['f1']:.4f} | Sent Micro-F1={mt['micro']['f1']:.4f} -> Saved!")
